# FOL Diagnostic: Sinh FOL + Z3 Parse Check

**Mục tiêu**: Chạy model FOL fine-tuned trên tập train, xuất CSV, rồi dùng Z3 parser kiểm tra từng formula để tìm:
1. FOL nào sinh sai cú pháp (Z3 parse fail)
2. FOL nào khác với gold label
3. Pattern lỗi phổ biến nhất

**Pipeline**: Load model → Sinh FOL (300 mẫu) → Xuất CSV → Z3 check → Báo cáo

## 1. Setup

In [ ]:
!pip install -q transformers accelerate bitsandbytes torch z3-solver

In [ ]:
import sys, os, json, re, ast, time
from pathlib import Path

import pandas as pd
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

# === Detect PROJECT_ROOT tự động ===
# Hoạt động trên cả local (Windows) và Modal (Linux /root/...)
# Tìm thư mục chứa "src/data/prompts.py" bằng cách thử các vị trí phổ biến
_cwd = Path(os.getcwd()).resolve()
PROJECT_ROOT = None
for _candidate in [
    _cwd,                                                    # nếu đang ở project root
    _cwd.parent,                                             # nếu đang ở notebooks/
    _cwd / "Logic_Based_Educational_Queries_Project",        # nếu đang ở repo root
    Path("/root/Logic_Based_Educational_Queries_Project"),   # Modal default
    Path("/root/Exact_2026_Laplace-s_Red_Devils/Logic_Based_Educational_Queries_Project"),
]:
    if (_candidate / "src" / "data" / "prompts.py").exists():
        PROJECT_ROOT = _candidate
        break

# Fallback: tìm bằng glob trong /root
if PROJECT_ROOT is None:
    for p in Path("/root").rglob("src/data/prompts.py"):
        PROJECT_ROOT = p.parent.parent.parent
        break

if PROJECT_ROOT is None:
    raise FileNotFoundError(
        f"Không tìm thấy project root (cần chứa src/data/prompts.py). "
        f"CWD = {_cwd}. Hãy cd vào thư mục project trước khi chạy."
    )

# Add src to Python path
sys.path.insert(0, str(PROJECT_ROOT / "src"))

# Config
MODEL_ID = "Laplaces-Red-Devils/fol-v04-cot-augmented-fol-pretrain-malls-qwen2.5-3"
MAX_SAMPLES = 300
MAX_NEW_TOKENS = 1024
RANDOM_SEED = 42
OUTPUT_CSV = PROJECT_ROOT / "notebooks" / "output" / "fol_diagnostic.csv"
OUTPUT_CSV.parent.mkdir(parents=True, exist_ok=True)

print(f"Project root: {PROJECT_ROOT}")
print(f"src in path:  {str(PROJECT_ROOT / 'src') in sys.path}")
print(f"Model: {MODEL_ID}")
print(f"Max samples: {MAX_SAMPLES}")
print(f"Output CSV: {OUTPUT_CSV}")

## 2. Load Model

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    trust_remote_code=True,
    quantization_config=BitsAndBytesConfig(load_in_8bit=True),
    device_map="auto",
)
model.eval()
print(f"Model loaded: {MODEL_ID}")

## 3. Load Prompt + Data

In [ ]:
from data.prompts import SYSTEM_PROMPT_FOL_SFT, USER_TEMPLATE_FOL_SFT, format_nl_block_numbered

# Đọc max_seq_length từ config
import yaml as _yaml
_cfg_path = PROJECT_ROOT / "configs" / "fol_model.yaml"
_MODEL_MAX_SEQ = 3072
if _cfg_path.is_file():
    _cfg = _yaml.safe_load(_cfg_path.read_text(encoding="utf-8")) or {}
    _MODEL_MAX_SEQ = int((_cfg.get("model") or {}).get("max_seq_length", 3072))

# Load train data
train_df = pd.read_csv(PROJECT_ROOT / "data" / "processed" / "train.csv")
print(f"Train data: {len(train_df)} rows")

unique_df = train_df.drop_duplicates(subset=["record_id"]).reset_index(drop=True)
print(f"Unique records (by record_id): {len(unique_df)}")

if MAX_SAMPLES < len(unique_df):
    sample_df = unique_df.sample(n=MAX_SAMPLES, random_state=RANDOM_SEED).reset_index(drop=True)
else:
    sample_df = unique_df.copy()
print(f"Sampled: {len(sample_df)} records")

sample_df["premises_nl_list"] = sample_df["premises_nl"].apply(ast.literal_eval)
sample_df["premises_fol_list"] = sample_df["premises_fol"].apply(ast.literal_eval)
sample_df["n_premises"] = sample_df["premises_nl_list"].apply(len)
print(f"Premises count: min={sample_df['n_premises'].min()}, max={sample_df['n_premises'].max()}, mean={sample_df['n_premises'].mean():.1f}")

# === Token Budget Analysis ===
print(f"\n--- Token Budget Analysis ---")
sys_tokens = len(tokenizer.encode(SYSTEM_PROMPT_FOL_SFT))
print(f"System prompt tokens: {sys_tokens}")
print(f"Model max_seq_length: {_MODEL_MAX_SEQ} (from config)")

token_counts = []
gold_fol_tokens = []
for _, row in sample_df.iterrows():
    nl_block = format_nl_block_numbered(row["premises_nl_list"])
    user_msg = USER_TEMPLATE_FOL_SFT.format(premises_nl=nl_block).strip()
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT_FOL_SFT.strip()},
        {"role": "user", "content": user_msg},
    ]
    full_prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    n_tokens = len(tokenizer.encode(full_prompt))
    token_counts.append(n_tokens)
    # Đo gold FOL output cần bao nhiêu tokens
    gold_json = json.dumps({"premises_fol": row["premises_fol_list"]}, ensure_ascii=False)
    gold_fol_tokens.append(len(tokenizer.encode(gold_json)))

sample_df["prompt_tokens"] = token_counts
sample_df["gold_fol_tokens"] = gold_fol_tokens
sample_df["total_needed"] = sample_df["prompt_tokens"] + sample_df["gold_fol_tokens"]
sample_df["remaining"] = _MODEL_MAX_SEQ - sample_df["prompt_tokens"]

print(f"Prompt tokens: min={min(token_counts)}, max={max(token_counts)}, mean={sum(token_counts)/len(token_counts):.0f}")
print(f"Gold FOL tokens: min={min(gold_fol_tokens)}, max={max(gold_fol_tokens)}, mean={sum(gold_fol_tokens)/len(gold_fol_tokens):.0f}")

# Thống kê thực tế: prompt + gold_fol có vượt max_seq_length không?
actually_truncated = (sample_df["total_needed"] > _MODEL_MAX_SEQ).sum()
print(f"\nSamples thực sự bị cắt (prompt+gold_fol > {_MODEL_MAX_SEQ}): {actually_truncated}/{len(sample_df)} ({actually_truncated/len(sample_df):.1%})")

# Top 5 dài nhất
top5 = sample_df.nlargest(5, "total_needed")[["record_id", "n_premises", "prompt_tokens", "gold_fol_tokens", "total_needed", "remaining"]]
print(f"\nTop 5 tốn token nhất:")
for _, r in top5.iterrows():
    status = "OK" if r["total_needed"] <= _MODEL_MAX_SEQ else "TRUNCATED"
    print(f"  record_id={int(r['record_id']):4d}  premises={int(r['n_premises']):2d}  "
          f"prompt={int(r['prompt_tokens'])}  gold_fol={int(r['gold_fol_tokens'])}  "
          f"total={int(r['total_needed'])}  remaining={int(r['remaining'])}  [{status}]")

## 3b. Kiểm tra training data có bị cắt không

Mô phỏng đúng quá trình training: system + user + assistant → tokenize → đếm tokens.
Nếu vượt `max_seq_length=2048` → training đã cắt assistant response → model học FOL bị cụt.

In [ ]:
# === Mô phỏng training: đếm tokens cho TOÀN BỘ sequence (system + user + assistant) ===
TRAIN_MAX_SEQ = 2048  # max_seq_length trong configs/fol_model.yaml khi train

# Load toàn bộ train data (không chỉ sample)
full_train_df = pd.read_csv(PROJECT_ROOT / "data" / "processed" / "train.csv")
full_train_unique = full_train_df.drop_duplicates(subset=["record_id"]).reset_index(drop=True)

truncated_records = []
all_seq_lengths = []

for _, row in full_train_unique.iterrows():
    premises_nl = ast.literal_eval(row["premises_nl"])
    premises_fol = ast.literal_eval(row["premises_fol"])
    record_id = row["record_id"]

    # Build đúng messages như training (system + user + assistant)
    nl_block = format_nl_block_numbered(premises_nl)
    user_msg = USER_TEMPLATE_FOL_SFT.format(premises_nl=nl_block).strip()
    assistant_msg = json.dumps({"premises_fol": premises_fol}, ensure_ascii=False)

    messages = [
        {"role": "system", "content": SYSTEM_PROMPT_FOL_SFT.strip()},
        {"role": "user", "content": user_msg},
        {"role": "assistant", "content": assistant_msg},
    ]

    # Tokenize giống training (không add_generation_prompt vì đã có assistant)
    full_text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)
    tokens = tokenizer.encode(full_text)
    seq_len = len(tokens)
    all_seq_lengths.append(seq_len)

    # Đếm riêng phần prompt (system + user) vs assistant
    prompt_messages = messages[:2]
    prompt_text = tokenizer.apply_chat_template(prompt_messages, tokenize=False, add_generation_prompt=True)
    prompt_tokens = len(tokenizer.encode(prompt_text))
    assistant_tokens = seq_len - prompt_tokens

    if seq_len > TRAIN_MAX_SEQ:
        cut_amount = seq_len - TRAIN_MAX_SEQ
        truncated_records.append({
            "record_id": record_id,
            "n_premises": len(premises_nl),
            "total_tokens": seq_len,
            "prompt_tokens": prompt_tokens,
            "assistant_tokens": assistant_tokens,
            "cut_tokens": cut_amount,
            "cut_pct": cut_amount / assistant_tokens * 100 if assistant_tokens > 0 else 0,
        })

# === Report ===
print("=" * 60)
print("  TRAINING DATA TRUNCATION ANALYSIS")
print("=" * 60)
print(f"\nTotal unique records: {len(full_train_unique)}")
print(f"max_seq_length (training): {TRAIN_MAX_SEQ}")

print(f"\n--- Sequence Length Distribution ---")
print(f"  Min:  {min(all_seq_lengths)} tokens")
print(f"  Max:  {max(all_seq_lengths)} tokens")
print(f"  Mean: {sum(all_seq_lengths)/len(all_seq_lengths):.0f} tokens")
print(f"  Median: {sorted(all_seq_lengths)[len(all_seq_lengths)//2]} tokens")

n_truncated = len(truncated_records)
print(f"\n--- Truncation ---")
print(f"  Records bị cắt (>{TRAIN_MAX_SEQ}): {n_truncated}/{len(full_train_unique)} ({n_truncated/len(full_train_unique):.1%})")

if n_truncated > 0:
    trunc_df = pd.DataFrame(truncated_records)
    print(f"  Tokens bị cắt: min={trunc_df['cut_tokens'].min()}, max={trunc_df['cut_tokens'].max()}, mean={trunc_df['cut_tokens'].mean():.0f}")
    print(f"  % assistant bị cắt: mean={trunc_df['cut_pct'].mean():.1f}%, max={trunc_df['cut_pct'].max():.1f}%")

    print(f"\n--- Top 10 records bị cắt nhiều nhất ---")
    worst = trunc_df.nlargest(10, "cut_tokens")
    for _, r in worst.iterrows():
        print(f"  record_id={int(r['record_id']):4d}  premises={int(r['n_premises']):2d}  "
              f"total={int(r['total_tokens'])}tok  prompt={int(r['prompt_tokens'])}tok  "
              f"assistant={int(r['assistant_tokens'])}tok  CUT={int(r['cut_tokens'])}tok ({r['cut_pct']:.0f}%)")

    # Histogram
    print(f"\n--- Distribution: bao nhiêu tokens bị cắt ---")
    bins_labels = [(1, 50, "1-50 tok"), (51, 200, "51-200 tok"), (201, 500, "201-500 tok"), (501, 9999, "500+ tok")]
    for lo, hi, label in bins_labels:
        count = len(trunc_df[(trunc_df["cut_tokens"] >= lo) & (trunc_df["cut_tokens"] <= hi)])
        bar = "#" * count
        print(f"  {label:12s} {count:4d} records  {bar}")
else:
    print("  Không có record nào bị cắt!")

# Records an toàn (không bị cắt)
safe = len(full_train_unique) - n_truncated
print(f"\n--- Kết luận ---")
if n_truncated == 0:
    print(f"  OK: Toàn bộ {safe} records fit trong {TRAIN_MAX_SEQ} tokens. Training data không bị cắt.")
elif n_truncated / len(full_train_unique) < 0.05:
    print(f"  NHẸE: Chỉ {n_truncated} records bị cắt ({n_truncated/len(full_train_unique):.1%}). Ảnh hưởng nhỏ.")
elif n_truncated / len(full_train_unique) < 0.30:
    print(f"  TRUNG BÌNH: {n_truncated} records bị cắt ({n_truncated/len(full_train_unique):.1%}). Cần tăng max_seq_length khi retrain.")
else:
    print(f"  NGHIÊM TRỌNG: {n_truncated} records bị cắt ({n_truncated/len(full_train_unique):.1%}). Model học từ data bị cụt!")
    print(f"  → Cần tăng max_seq_length lên ít nhất {max(all_seq_lengths)} hoặc rút ngắn system prompt.")

print("=" * 60)

## 4. Generate FOL for all samples

In [ ]:
BATCH_SIZE = 14  # Chạy batch để tận dụng GPU

# Left-padding cho generation
tokenizer.padding_side = "left"
if tokenizer.pad_token_id is None:
    tokenizer.pad_token_id = tokenizer.eos_token_id

# Đọc max_seq_length từ config YAML (khớp với training)
import yaml as _yaml
_cfg_path = PROJECT_ROOT / "configs" / "fol_model.yaml"
MODEL_MAX_SEQ = 3072  # default
if _cfg_path.is_file():
    _cfg = _yaml.safe_load(_cfg_path.read_text(encoding="utf-8")) or {}
    MODEL_MAX_SEQ = int((_cfg.get("model") or {}).get("max_seq_length", 3072))


def build_prompt(premises_nl: list[str]) -> str:
    """Tạo prompt cho 1 sample."""
    nl_block = format_nl_block_numbered(premises_nl)
    user_msg = USER_TEMPLATE_FOL_SFT.format(premises_nl=nl_block).strip()
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT_FOL_SFT.strip()},
        {"role": "user", "content": user_msg},
    ]
    return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)


def generate_fol_batch(prompts: list[str]) -> list[str]:
    """Sinh FOL cho nhiều prompts cùng lúc trên GPU."""
    inputs = tokenizer(
        prompts,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=MODEL_MAX_SEQ,
    ).to(model.device)

    prompt_len = inputs["input_ids"].shape[1]
    # Dynamic budget: tokens còn lại cho output
    budget = max(MODEL_MAX_SEQ - prompt_len, 128)
    actual_max_new = min(budget, MAX_NEW_TOKENS)

    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=actual_max_new,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
            repetition_penalty=1.2,
        )

    # Decode từng sample
    results = []
    for row in out:
        gen_ids = row[prompt_len:]
        text = tokenizer.decode(gen_ids, skip_special_tokens=True).strip()
        results.append(text)
    return results, prompt_len, actual_max_new


def parse_fol_json(text: str) -> list[str] | None:
    """Parse JSON output -> list[str]. 3 cấp fallback."""
    if not text or not text.strip():
        return None

    # Attempt 1: JSON hợp lệ
    match = re.search(r"\{.*\}", text, re.DOTALL)
    if match:
        try:
            parsed = json.loads(match.group())
            if "premises_fol" in parsed and isinstance(parsed["premises_fol"], list):
                return [str(f).strip() for f in parsed["premises_fol"]]
        except json.JSONDecodeError:
            pass

    # Attempt 2: Repair JSON bị cắt
    start = text.find('{"premises_fol"')
    if start == -1:
        start = text.find('"premises_fol"')
    if start >= 0:
        fol_strings = re.findall(r'"([^"]*)"', text[start:])
        values = [s for s in fol_strings if s != "premises_fol" and len(s) > 1]
        if values:
            return [v.strip() for v in values]

    # Attempt 3: Fallback — tìm FOL lines
    lines = []
    for line in text.split("\n"):
        line = line.strip().lstrip("0123456789.)-  ")
        if any(c in line for c in "∀∃→∧∨¬↔") or re.match(r"[A-Z]\w*\(", line):
            lines.append(line)
    if lines:
        return lines

    return None


print(f"Functions ready.")
print(f"BATCH_SIZE: {BATCH_SIZE}")
print(f"MODEL_MAX_SEQ: {MODEL_MAX_SEQ} (from {_cfg_path.name})")
print(f"MAX_NEW_TOKENS: {MAX_NEW_TOKENS}")

In [ ]:
# === Generate FOL — BATCHED ===
results = []
t_start = time.time()
n_total = len(sample_df)
debug_fail_count = 0

for batch_start in range(0, n_total, BATCH_SIZE):
    batch_end = min(batch_start + BATCH_SIZE, n_total)
    batch_rows = sample_df.iloc[batch_start:batch_end]

    # Build prompts
    prompts = [build_prompt(row["premises_nl_list"]) for _, row in batch_rows.iterrows()]

    # Generate batch
    t0 = time.time()
    raw_outputs, prompt_len, actual_max_new = generate_fol_batch(prompts)
    batch_time = time.time() - t0

    # Process từng sample
    for i, (_, row) in enumerate(batch_rows.iterrows()):
        premises_nl = row["premises_nl_list"]
        gold_fol = row["premises_fol_list"]
        record_id = row["record_id"]
        raw_output = raw_outputs[i]

        pred_fol = parse_fol_json(raw_output)
        json_parse_ok = pred_fol is not None
        if pred_fol is None:
            pred_fol = []

        n_gold = len(gold_fol)
        n_pred = len(pred_fol)

        for p_idx in range(max(n_gold, n_pred)):
            nl = premises_nl[p_idx] if p_idx < len(premises_nl) else "(no NL)"
            g = gold_fol[p_idx] if p_idx < n_gold else "(missing in gold)"
            p = pred_fol[p_idx] if p_idx < n_pred else "(missing in pred)"
            exact_match = g.strip() == p.strip() if p_idx < n_gold and p_idx < n_pred else False

            results.append({
                "record_id": record_id,
                "premise_idx": p_idx,
                "premise_nl": nl,
                "gold_fol": g,
                "predicted_fol": p,
                "exact_match": exact_match,
                "json_parse_ok": json_parse_ok,
                "n_gold": n_gold,
                "n_pred": n_pred,
                "count_match": n_gold == n_pred,
                "gen_time_s": batch_time / len(batch_rows),
                "prompt_tokens": prompt_len,
                "max_new_tokens_used": actual_max_new,
                "raw_output": raw_output if not json_parse_ok else "",
            })

        # Debug fail
        if not json_parse_ok and debug_fail_count < 5:
            debug_fail_count += 1
            print(f"  >>> DEBUG FAIL #{debug_fail_count} record_id={record_id} "
                  f"prompt={prompt_len}tok max_new={actual_max_new}tok:")
            print(f"  >>> {raw_output[:500]}")
            print()

    # Progress
    done = batch_end
    elapsed = time.time() - t_start
    eta = (elapsed / done) * (n_total - done) if done > 0 else 0
    batch_json_ok = sum(1 for r in raw_outputs if parse_fol_json(r) is not None)
    print(f"[{done}/{n_total}] batch {batch_time:.1f}s ({batch_time/len(batch_rows):.1f}s/sample) "
          f"json_ok={batch_json_ok}/{len(batch_rows)} prompt={prompt_len}tok budget={actual_max_new}tok "
          f"ETA={eta/60:.1f}m")

total_time = time.time() - t_start
print(f"\nDone! {n_total} records, {len(results)} premises, {total_time/60:.1f} minutes")
print(f"Speed: {total_time/n_total:.1f}s/sample (batch={BATCH_SIZE})")

## 5. Export CSV

In [ ]:
df_results = pd.DataFrame(results)
df_results.to_csv(OUTPUT_CSV, index=False, encoding="utf-8-sig")
print(f"Saved: {OUTPUT_CSV}")
print(f"Total rows: {len(df_results)}")
print(f"\nQuick stats:")
print(f"  JSON parse OK:    {df_results['json_parse_ok'].mean():.1%}")
print(f"  Count match:      {df_results.groupby('record_id')['count_match'].first().mean():.1%}")
print(f"  Exact match:      {df_results['exact_match'].mean():.1%}")
df_results.head(10)

## 6. Z3 Parse Check

Dùng parser từ `src/evaluation/fol_parser.py` để check từng FOL formula có parse được không.

In [ ]:
from evaluation.fol_parser import parse_fol, FOLParseError
from evaluation.fol_z3_translator import safe_fol_string_to_z3

def check_z3_parse(fol_str: str) -> tuple[bool, str]:
    """Check if a FOL string can be parsed and translated to Z3.
    Returns (success: bool, error_msg: str).
    """
    if not fol_str or fol_str.startswith("(missing"):
        return False, "empty_or_missing"
    
    # Step 1: Parse to AST
    try:
        ast_node = parse_fol(fol_str)
    except FOLParseError as e:
        return False, f"parse_error: {e}"
    except Exception as e:
        return False, f"unexpected_error: {e}"
    
    # Step 2: Translate AST to Z3
    cache = {}
    z3_expr = safe_fol_string_to_z3(fol_str, cache)
    if z3_expr is None:
        return False, "z3_translation_failed"
    
    return True, ""


# Run Z3 check on all predicted FOL
print("Running Z3 parse check on predicted FOL...")
z3_results_pred = []
for _, row in df_results.iterrows():
    ok, err = check_z3_parse(row["predicted_fol"])
    z3_results_pred.append({"z3_parse_ok": ok, "z3_error_msg": err})

z3_pred_df = pd.DataFrame(z3_results_pred)
df_results["z3_parse_ok"] = z3_pred_df["z3_parse_ok"]
df_results["z3_error_msg"] = z3_pred_df["z3_error_msg"]

# Also check gold FOL (to see if gold itself has issues)
print("Running Z3 parse check on gold FOL...")
z3_results_gold = []
for _, row in df_results.iterrows():
    ok, err = check_z3_parse(row["gold_fol"])
    z3_results_gold.append({"gold_z3_parse_ok": ok, "gold_z3_error_msg": err})

z3_gold_df = pd.DataFrame(z3_results_gold)
df_results["gold_z3_parse_ok"] = z3_gold_df["gold_z3_parse_ok"]
df_results["gold_z3_error_msg"] = z3_gold_df["gold_z3_error_msg"]

# Save updated CSV
df_results.to_csv(OUTPUT_CSV, index=False, encoding="utf-8-sig")
print(f"\nUpdated CSV saved: {OUTPUT_CSV}")

## 7. Diagnostic Report

In [ ]:
total = len(df_results)
pred_not_missing = df_results[~df_results["predicted_fol"].str.startswith("(missing")]
total_valid = len(pred_not_missing)

print("=" * 60)
print("  FOL DIAGNOSTIC REPORT")
print("=" * 60)

# --- Overall stats ---
print(f"\n--- Overall ---")
print(f"Total premises:          {total}")
print(f"Valid predictions:       {total_valid} ({total_valid/total:.1%})")
print(f"Missing predictions:     {total - total_valid}")

n_records = df_results["record_id"].nunique()
json_ok_records = df_results.groupby("record_id")["json_parse_ok"].first()
print(f"\n--- JSON Parse (model output -> list) ---")
print(f"Records with valid JSON: {json_ok_records.sum()}/{n_records} ({json_ok_records.mean():.1%})")

count_match = df_results.groupby("record_id")["count_match"].first()
print(f"\n--- Count Match (n_gold == n_pred) ---")
print(f"Records with count match: {count_match.sum()}/{n_records} ({count_match.mean():.1%})")

# --- Z3 Parse ---
print(f"\n--- Z3 Parse (predicted FOL -> Z3) ---")
z3_ok = pred_not_missing["z3_parse_ok"].sum()
print(f"Z3 parse OK:  {z3_ok}/{total_valid} ({z3_ok/total_valid:.1%})")
print(f"Z3 parse FAIL: {total_valid - z3_ok}/{total_valid} ({(total_valid-z3_ok)/total_valid:.1%})")

print(f"\n--- Z3 Parse (gold FOL -> Z3) ---")
gold_valid = df_results[~df_results["gold_fol"].str.startswith("(missing")]
gold_z3_ok = gold_valid["gold_z3_parse_ok"].sum()
print(f"Gold Z3 parse OK:  {gold_z3_ok}/{len(gold_valid)} ({gold_z3_ok/len(gold_valid):.1%})")

# --- Exact Match ---
print(f"\n--- Exact Match (pred == gold string) ---")
em = df_results["exact_match"].sum()
print(f"Exact match: {em}/{total} ({em/total:.1%})")

print(f"\n" + "=" * 60)

In [ ]:
# --- Z3 Parse Error Analysis ---
print("\n--- Top Z3 Parse Error Patterns (predicted FOL) ---\n")

failed = pred_not_missing[~pred_not_missing["z3_parse_ok"]].copy()

if len(failed) > 0:
    # Categorize errors
    def categorize_error(err_msg: str, fol: str) -> str:
        if "parse_error" in err_msg:
            if "Extra tokens" in err_msg:
                return "extra_tokens (unbalanced parens or trailing junk)"
            if "Expected variable" in err_msg:
                return "bad_variable (multi-char var after quantifier)"
            if "Unexpected token" in err_msg:
                return "unexpected_token"
            if "Unexpected end" in err_msg:
                return "unexpected_eof"
            return "other_parse_error"
        if "z3_translation" in err_msg:
            return "z3_translation_failed"
        if "empty" in err_msg:
            return "empty"
        return "unknown"

    failed["error_category"] = failed.apply(
        lambda r: categorize_error(r["z3_error_msg"], r["predicted_fol"]), axis=1
    )

    # Count by category
    cat_counts = failed["error_category"].value_counts()
    for cat, count in cat_counts.items():
        pct = count / total_valid * 100
        print(f"  {cat:50s} {count:5d} ({pct:.1f}%)")

    print(f"\n--- Sample failures per category ---\n")
    for cat in cat_counts.index[:5]:  # Top 5 categories
        samples = failed[failed["error_category"] == cat].head(3)
        print(f"\n  [{cat}]")
        for _, s in samples.iterrows():
            print(f"    FOL:   {s['predicted_fol'][:100]}")
            print(f"    Error: {s['z3_error_msg'][:100]}")
            print()
else:
    print("  No Z3 parse failures!")

In [ ]:
# --- Gold FOL parse failures (if any) ---
print("\n--- Gold FOL that ALSO fail Z3 parse ---\n")

gold_failed = gold_valid[~gold_valid["gold_z3_parse_ok"]]
if len(gold_failed) > 0:
    print(f"Gold Z3 failures: {len(gold_failed)}/{len(gold_valid)} ({len(gold_failed)/len(gold_valid):.1%})")
    print()
    for _, s in gold_failed.head(10).iterrows():
        print(f"  record_id={s['record_id']} premise_idx={s['premise_idx']}")
        print(f"  Gold FOL: {s['gold_fol'][:120]}")
        print(f"  Error:    {s['gold_z3_error_msg'][:120]}")
        print()
else:
    print("  All gold FOL parsed successfully by Z3.")

In [ ]:
# --- Per-record summary: how many premises parse vs fail ---
print("\n--- Per-record Z3 parse rate ---\n")

record_stats = pred_not_missing.groupby("record_id").agg(
    total_premises=("z3_parse_ok", "count"),
    z3_ok=("z3_parse_ok", "sum"),
    exact_matches=("exact_match", "sum"),
).reset_index()
record_stats["z3_fail"] = record_stats["total_premises"] - record_stats["z3_ok"]
record_stats["z3_rate"] = record_stats["z3_ok"] / record_stats["total_premises"]

# Distribution
print("Z3 parse rate distribution:")
bins = [0, 0.5, 0.8, 0.99, 1.0]
labels = ["< 50%", "50-80%", "80-99%", "100%"]
record_stats["rate_bin"] = pd.cut(record_stats["z3_rate"], bins=bins, labels=labels, include_lowest=True)
dist = record_stats["rate_bin"].value_counts().sort_index()
for label, count in dist.items():
    pct = count / len(record_stats) * 100
    bar = '#' * int(pct / 2)
    print(f"  {label:10s} {count:4d} records ({pct:5.1f}%) {bar}")

print(f"\nWorst records (most Z3 failures):")
worst = record_stats.nlargest(10, "z3_fail")
for _, r in worst.iterrows():
    print(f"  record_id={int(r['record_id']):4d}  "
          f"z3_ok={int(r['z3_ok'])}/{int(r['total_premises'])}  "
          f"z3_fail={int(r['z3_fail'])}  "
          f"exact_match={int(r['exact_matches'])}")

## 8. Predicate Collision Check

Kiểm tra xem predicted FOL có bị trùng predicate name cho 2 concepts khác nhau không.

In [ ]:
import re

def extract_predicates(fol_str: str) -> set[str]:
    """Extract predicate names from FOL string."""
    # Match: Name( or Name(x
    return set(re.findall(r'([A-Za-z_][A-Za-z0-9_]*)\s*\(', fol_str))


print("--- Predicate Analysis per Record ---\n")

pred_analysis = []
for record_id, group in pred_not_missing.groupby("record_id"):
    all_preds = set()
    for fol in group["predicted_fol"]:
        all_preds.update(extract_predicates(fol))

    gold_preds = set()
    for fol in group["gold_fol"]:
        gold_preds.update(extract_predicates(fol))

    pred_analysis.append({
        "record_id": record_id,
        "n_pred_predicates": len(all_preds),
        "n_gold_predicates": len(gold_preds),
        "pred_predicates": all_preds,
        "gold_predicates": gold_preds,
    })

pred_df = pd.DataFrame(pred_analysis)
pred_df["pred_vs_gold_ratio"] = pred_df["n_pred_predicates"] / pred_df["n_gold_predicates"].clip(lower=1)

# Records where predicted has far fewer predicates than gold (possible collision)
collision_suspects = pred_df[pred_df["pred_vs_gold_ratio"] < 0.6]
print(f"Collision suspects (pred predicates < 60% of gold): {len(collision_suspects)}/{len(pred_df)}")
if len(collision_suspects) > 0:
    for _, r in collision_suspects.head(5).iterrows():
        print(f"\n  record_id={int(r['record_id'])}:")
        print(f"    Gold predicates ({r['n_gold_predicates']}): {r['gold_predicates']}")
        print(f"    Pred predicates ({r['n_pred_predicates']}): {r['pred_predicates']}")

# Records where predicted has far more predicates (no reuse)
no_reuse = pred_df[pred_df["pred_vs_gold_ratio"] > 2.0]
print(f"\nNo-reuse suspects (pred predicates > 200% of gold): {len(no_reuse)}/{len(pred_df)}")
if len(no_reuse) > 0:
    for _, r in no_reuse.head(5).iterrows():
        print(f"\n  record_id={int(r['record_id'])}:")
        print(f"    Gold predicates ({r['n_gold_predicates']}): {r['gold_predicates']}")
        print(f"    Pred predicates ({r['n_pred_predicates']}): {r['pred_predicates']}")

## 9. Summary

In [ ]:
print("=" * 60)
print("  FINAL SUMMARY")
print("=" * 60)
print(f"""  
  Model:            {MODEL_ID}
  Samples:           {len(sample_df)} records, {total} premises
  
  JSON parse rate:   {json_ok_records.mean():.1%}
  Count match rate:  {count_match.mean():.1%}
  Exact match rate:  {em/total:.1%}
  
  Z3 parse (pred):   {z3_ok}/{total_valid} ({z3_ok/total_valid:.1%})
  Z3 parse (gold):   {gold_z3_ok}/{len(gold_valid)} ({gold_z3_ok/len(gold_valid):.1%})
  
  Collision suspects: {len(collision_suspects)}
  No-reuse suspects:  {len(no_reuse)}
  
  Output CSV:        {OUTPUT_CSV}
""")
print("=" * 60)